In [3]:
import requests
import pandas as pd
import time
from datetime import datetime


# -------------------------------
# Helper: Retry logic
# -------------------------------
def make_request(url, params, city_name):
    for attempt in range(3):
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                raise Exception(f"HTTP {response.status_code}")

            data = response.json()

            return data

        except Exception as e:
            print(f"[{city_name}] Attempt {attempt+1} failed: {e}")

            # Exponential backoff: 1s → 2s → 4s
            time.sleep(2 ** attempt)

    raise Exception(f"[{city_name}] Failed after 3 retries")


# -------------------------------
# Fetch Historical Data
# -------------------------------
def fetch_historical(city_name, latitude, longitude, start_date, end_date, variables):
    
    # ✅ Date validation
    if start_date >= end_date:
        raise ValueError("start_date must be before end_date")

    today = datetime.today().strftime("%Y-%m-%d")
    if end_date > today:
        raise ValueError("end_date cannot be in the future")

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join(variables)
    }

    data = make_request(url, params, city_name)

    # ✅ Check response
    if "daily" not in data:
        raise ValueError(f"[{city_name}] Malformed response: {data}")

    df = pd.DataFrame(data["daily"])

    # ✅ Basic validation
    if df.empty:
        raise ValueError(f"[{city_name}] Empty dataset returned")

    df["time"] = pd.to_datetime(df["time"])
    df["city"] = city_name

    return df


# -------------------------------
# Fetch Forecast Data
# -------------------------------
def fetch_forecast(city_name, latitude, longitude, variables):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": ",".join(variables)
    }

    data = make_request(url, params, city_name)

    if "daily" not in data:
        raise ValueError(f"[{city_name}] Malformed forecast response: {data}")

    df = pd.DataFrame(data["daily"])

    if df.empty:
        raise ValueError(f"[{city_name}] Empty forecast data")

    df["time"] = pd.to_datetime(df["time"])
    df["city"] = city_name

    return df


# -------------------------------
# Fetch All Cities
# -------------------------------
def fetch_all_cities(cities_config, start_date, end_date, variables):

    all_data = {}

    for city in cities_config:
        name = city["name"]
        lat = city["latitude"]
        lon = city["longitude"]

        print(f"Fetching historical data for {name}...")

        df = fetch_historical(
            name,
            lat,
            lon,
            start_date,
            end_date,
            variables
        )

        all_data[name] = df

        print(f"{name}: {len(df)} rows")

    return all_data

In [5]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from ingestion import fetch_historical

df = fetch_historical(
    "Baku",
    40.41,
    49.87,
    "2020-01-01",
    "2022-12-31",
    ["temperature_2m_max", "precipitation_sum"]
)

df.head()

,time,temperature_2m_max,precipitation_sum,city
0,2020-01-01,11.4,0.0,Baku
1,2020-01-02,8.6,0.3,Baku
2,2020-01-03,7.6,1.4,Baku
3,2020-01-04,8.4,0.2,Baku
4,2020-01-05,7.7,2.6,Baku


In [7]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from ingestion import fetch_historical
from config import CITIES, START_DATE, END_DATE, DAILY_VARIABLES

city = CITIES[0]

df = fetch_historical(
    city_name=city["name"],
    latitude=city["latitude"],
    longitude=city["longitude"],
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES
)

df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,apparent_temperature_max,relative_humidity_2m_mean,weather_code,city
0,2020-01-01,11.4,4.4,0.0,11.4,8.8,88,3,Baku
1,2020-01-02,8.6,2.7,0.3,33.0,3.2,86,51,Baku
2,2020-01-03,7.6,3.4,1.4,21.7,3.3,84,51,Baku
3,2020-01-04,8.4,5.7,0.2,22.7,4.6,81,51,Baku
4,2020-01-05,7.7,5.8,2.6,12.6,6.4,94,53,Baku


In [9]:
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df["time"].min(), df["time"].max())

(2192, 9)
Index(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'wind_speed_10m_max', 'apparent_temperature_max',
       'relative_humidity_2m_mean', 'weather_code', 'city'],
      dtype='object')
time                         datetime64[ns]
temperature_2m_max                  float64
temperature_2m_min                  float64
precipitation_sum                   float64
wind_speed_10m_max                  float64
apparent_temperature_max            float64
relative_humidity_2m_mean             int64
weather_code                          int64
city                                 object
dtype: object
2020-01-01 00:00:00 2025-12-31 00:00:00


In [11]:
from ingestion import fetch_all_cities

data = fetch_all_cities(
    cities_config=CITIES,
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES
)

for city_name, df in data.items():
    print(f"{city_name}: {df.shape}, {df['time'].min()} → {df['time'].max()}")

Baku: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Lankaran: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Guba: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Gabala: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00
Shaki: (2192, 9), 2020-01-01 00:00:00 → 2025-12-31 00:00:00


-------------------------------------

In [14]:
import pandas as pd

historical_df = pd.concat(data.values(), ignore_index=True)
historical_df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,apparent_temperature_max,relative_humidity_2m_mean,weather_code,city
0,2020-01-01,11.4,4.4,0.0,11.4,8.8,88,3,Baku
1,2020-01-02,8.6,2.7,0.3,33.0,3.2,86,51,Baku
2,2020-01-03,7.6,3.4,1.4,21.7,3.3,84,51,Baku
3,2020-01-04,8.4,5.7,0.2,22.7,4.6,81,51,Baku
4,2020-01-05,7.7,5.8,2.6,12.6,6.4,94,53,Baku


In [16]:
total_rows = len(historical_df)
print("Total rows ingested:", total_rows)

Total rows ingested: 10960


In [18]:
historical_df["city"].value_counts()

city
Baku        2192
Lankaran    2192
Guba        2192
Gabala      2192
Shaki       2192
Name: count, dtype: int64

In [20]:
for city, city_df in historical_df.groupby("city"):
    city_df = city_df.sort_values("time")
    
    expected_dates = pd.date_range(
        start=city_df["time"].min(),
        end=city_df["time"].max(),
        freq="D"
    )
    
    actual_dates = pd.DatetimeIndex(city_df["time"])
    missing_dates = expected_dates.difference(actual_dates)
    
    print(f"\n{city}")
    print("Expected days:", len(expected_dates))
    print("Actual rows   :", len(city_df))
    print("Missing days  :", len(missing_dates))
    
    if len(missing_dates) > 0:
        print("First missing dates:", missing_dates[:10])


Baku
Expected days: 2192
Actual rows   : 2192
Missing days  : 0

Gabala
Expected days: 2192
Actual rows   : 2192
Missing days  : 0

Guba
Expected days: 2192
Actual rows   : 2192
Missing days  : 0

Lankaran
Expected days: 2192
Actual rows   : 2192
Missing days  : 0

Shaki
Expected days: 2192
Actual rows   : 2192
Missing days  : 0


In [22]:
null_counts = historical_df.isnull().sum().sort_values(ascending=False)
print(null_counts)

time                         0
temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
wind_speed_10m_max           0
apparent_temperature_max     0
relative_humidity_2m_mean    0
weather_code                 0
city                         0
dtype: int64


In [24]:
variable_nulls = historical_df.drop(columns=["time", "city"]).isnull().sum().sort_values(ascending=False)
print(variable_nulls)

temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
wind_speed_10m_max           0
apparent_temperature_max     0
relative_humidity_2m_mean    0
weather_code                 0
dtype: int64


In [37]:
historical_df.groupby("city").apply(
    lambda x: x.isnull().sum(),
    include_groups=False
)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,apparent_temperature_max,relative_humidity_2m_mean,weather_code
city,,,,,,,,
Baku,0,0,0,0,0,0,0,0
Gabala,0,0,0,0,0,0,0,0
Guba,0,0,0,0,0,0,0,0
Lankaran,0,0,0,0,0,0,0,0
Shaki,0,0,0,0,0,0,0,0


In [28]:
from config import START_DATE, END_DATE
print("Requested start:", START_DATE)
print("Requested end  :", END_DATE)
actual_range = historical_df.groupby("city").agg(
    actual_start=("time", "min"),
    actual_end=("time", "max"),
    rows=("time", "count")
)

print(actual_range)

Requested start: 2020-01-01
Requested end  : 2025-12-31
         actual_start actual_end  rows
city                                  
Baku       2020-01-01 2025-12-31  2192
Gabala     2020-01-01 2025-12-31  2192
Guba       2020-01-01 2025-12-31  2192
Lankaran   2020-01-01 2025-12-31  2192
Shaki      2020-01-01 2025-12-31  2192


In [30]:
actual_range["requested_start"] = pd.to_datetime(START_DATE)
actual_range["requested_end"] = pd.to_datetime(END_DATE)
actual_range["start_matches"] = actual_range["actual_start"] == actual_range["requested_start"]
actual_range["end_matches"] = actual_range["actual_end"] == actual_range["requested_end"]
actual_range

,actual_start,actual_end,rows,requested_start,requested_end,start_matches,end_matches
city,,,,,,,
Baku,2020-01-01,2025-12-31,2192,2020-01-01,2025-12-31,True,True
Gabala,2020-01-01,2025-12-31,2192,2020-01-01,2025-12-31,True,True
Guba,2020-01-01,2025-12-31,2192,2020-01-01,2025-12-31,True,True
Lankaran,2020-01-01,2025-12-31,2192,2020-01-01,2025-12-31,True,True
Shaki,2020-01-01,2025-12-31,2192,2020-01-01,2025-12-31,True,True


In [32]:
duplicates = historical_df.duplicated(subset=["city", "time"]).sum()
print("Duplicate city-date rows:", duplicates)

Duplicate city-date rows: 0


In [34]:
audit_rows = []

for city, city_df in historical_df.groupby("city"):
    city_df = city_df.sort_values("time")
    
    expected_dates = pd.date_range(city_df["time"].min(), city_df["time"].max(), freq="D")
    missing_dates = expected_dates.difference(pd.DatetimeIndex(city_df["time"]))
    
    row = {
        "city": city,
        "rows": len(city_df),
        "start_date": city_df["time"].min(),
        "end_date": city_df["time"].max(),
        "missing_days": len(missing_dates),
        "duplicate_city_date_rows": city_df.duplicated(subset=["time"]).sum(),
        "total_nulls": city_df.isnull().sum().sum()
    }
    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)
audit_df

,city,rows,start_date,end_date,missing_days,duplicate_city_date_rows,total_nulls
0,Baku,2192,2020-01-01,2025-12-31,0,0,0
1,Gabala,2192,2020-01-01,2025-12-31,0,0,0
2,Guba,2192,2020-01-01,2025-12-31,0,0,0
3,Lankaran,2192,2020-01-01,2025-12-31,0,0,0
4,Shaki,2192,2020-01-01,2025-12-31,0,0,0


The data audit serves as the first trust checkpoint for the ingestion pipeline. I verified the total row counts, checked whether the daily date sequence contained gaps, measured null values across variables, and compared the actual returned date range with the requested range. If all cities show complete date coverage, zero missing days, and minimal or no null values, then the ingestion pipeline can be considered reliable enough for the next project stage. Any issues found at this stage should be resolved before feature engineering or modeling.

In [39]:
(historical_df["temperature_2m_max"] < historical_df["temperature_2m_min"]).sum()

0

In [41]:
historical_df

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,apparent_temperature_max,relative_humidity_2m_mean,weather_code,city
0,2020-01-01,11.4,4.4,0.0,11.4,8.8,88,3,Baku
1,2020-01-02,8.6,2.7,0.3,33.0,3.2,86,51,Baku
2,2020-01-03,7.6,3.4,1.4,21.7,3.3,84,51,Baku
3,2020-01-04,8.4,5.7,0.2,22.7,4.6,81,51,Baku
4,2020-01-05,7.7,5.8,2.6,12.6,6.4,94,53,Baku
...,...,...,...,...,...,...,...,...,...
10955,2025-12-27,7.5,-1.3,0.7,11.3,3.9,78,71,Shaki
10956,2025-12-28,1.0,-4.2,7.3,13.4,-1.5,92,75,Shaki
10957,2025-12-29,4.4,-2.6,0.0,21.3,0.5,62,3,Shaki
10958,2025-12-30,4.6,-2.3,0.0,22.0,-1.2,56,1,Shaki
